<a href="https://colab.research.google.com/github/yashlotekar28/campusknowledgeassistantusingrag/blob/main/Minor_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# MIT AOE RAG PROJECT
# BLOCK 1: INSTALL AND IMPORT LIBRARIES
# ============================================

!pip -q install pymupdf chromadb sentence-transformers google-genai pytesseract
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

import os
import re
import hashlib
import time
import json

import pymupdf
import chromadb
import pytesseract

from PIL import Image
from sentence_transformers import SentenceTransformer
from google import genai

from google.colab import files, drive
from getpass import getpass

print("All required libraries are ready!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 2: UPLOAD AND EXTRACT PDF TEXT
# ============================================

uploaded = files.upload()

documents = []

for filename in uploaded.keys():

    print(f"\nProcessing PDF: {filename}")

    pdf = pymupdf.open(filename)

    for page_index, page in enumerate(pdf):

        # First, try normal text extraction
        text = page.get_text("text").strip()

        # If the page has very little text, use OCR
        if len(text) < 50:

            print(f"Using OCR for page {page_index + 1}")

            pix = page.get_pixmap(
                matrix=pymupdf.Matrix(2, 2),
                alpha=False
            )

            image_path = f"/tmp/ocr_page_{page_index + 1}.png"
            pix.save(image_path)

            image = Image.open(image_path)

            text = pytesseract.image_to_string(
                image,
                lang="eng"
            ).strip()

        # Save each page with its source information
        if text:
            documents.append({
                "text": text,
                "source": filename,
                "page_number": page_index + 1
            })

    pdf.close()

print("\nPDF extraction completed!")
print("Total pages with extracted text:", len(documents))
print("Uploaded PDF files:", len(uploaded))

# Preview the first extracted page
if documents:
    print("\nFirst page preview:")
    print("Source:", documents[0]["source"])
    print("Page:", documents[0]["page_number"])
    print(documents[0]["text"][:1000])
else:
    print("No text extracted. Check whether the PDFs contain readable text or images.")


KeyboardInterrupt: 

In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 3: STRUCTURE-AWARE CHUNKING
# ============================================

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

chunks = []

def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Split the text into paragraphs
    paragraphs = re.split(r"\n\s*\n", text)

    # If paragraph breaks are missing, use individual lines
    if len(paragraphs) == 1:
        paragraphs = text.splitlines()

    result = []
    current_chunk = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        # Split very long paragraphs into smaller pieces
        if len(paragraph) > chunk_size:
            sentences = re.split(r"(?<=[.!?])\s+", paragraph)
        else:
            sentences = [paragraph]

        for sentence in sentences:
            sentence = sentence.strip()

            if not sentence:
                continue

            # Add the sentence if it fits in the current chunk
            if len(current_chunk) + len(sentence) + 1 <= chunk_size:
                current_chunk = (current_chunk + " " + sentence).strip()

            else:
                # Save the current chunk
                if current_chunk:
                    result.append(current_chunk)

                # Carry a small overlap into the next chunk
                if current_chunk and overlap > 0:
                    current_chunk = current_chunk[-overlap:].strip()
                else:
                    current_chunk = ""

                # Handle a sentence that is itself too long
                while len(sentence) > chunk_size:
                    result.append(sentence[:chunk_size])
                    sentence = sentence[chunk_size:]

                current_chunk = (current_chunk + " " + sentence).strip()

    if current_chunk:
        result.append(current_chunk)

    return result


# Process each extracted PDF page separately
for document in documents:

    page_chunks = split_into_chunks(document["text"])

    for chunk_index, chunk_text in enumerate(page_chunks, start=1):
        chunks.append({
            "text": chunk_text,
            "source": document["source"],
            "page_number": document["page_number"],
            "chunk_id": (
                f'{document["source"]}_'
                f'page_{document["page_number"]}_'
                f'chunk_{chunk_index}'
            )
        })


# Show the results
print("Chunking completed!")
print("Total extracted pages:", len(documents))
print("Total chunks created:", len(chunks))

if chunks:
    print("\nFirst chunk preview:")
    print("Source:", chunks[0]["source"])
    print("Page:", chunks[0]["page_number"])
    print("Chunk ID:", chunks[0]["chunk_id"])
    print("Text:")
    print(chunks[0]["text"])
else:
    print("No chunks created. Check Block 2 output.")


NameError: name 'documents' is not defined

In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 4: EMBEDDINGS AND VECTOR DATABASE
# ============================================

import hashlib
import json
import os

# 1. Connect Google Drive
drive.mount("/content/drive")

# 2. Set persistent database location
DB_PATH = "/content/drive/MyDrive/MIT_AOE_RAG_DB"
os.makedirs(DB_PATH, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=DB_PATH)

# 3. Create or access the collection
collection = chroma_client.get_or_create_collection(
    name="mit_aoe_knowledge",
    metadata={"hnsw:space": "cosine"}
)

# 4. Load the embedding model once
print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")


# ============================================
# 5. IDENTIFY DOCUMENTS BY THEIR CONTENT
# ============================================

# Create a stable fingerprint for each complete PDF.
# Identical PDFs with different filenames get the same fingerprint.

document_fingerprints = {}

for document in documents:

    source = document["source"]

    # Collect all extracted pages belonging to this PDF
    if source not in document_fingerprints:
        document_fingerprints[source] = []

    document_fingerprints[source].append(
        (
            document["page_number"],
            document["text"].strip()
        )
    )

fingerprint_by_source = {}

for source, page_data in document_fingerprints.items():

    page_data.sort(key=lambda item: item[0])

    full_text = "\n".join(
        f"PAGE {page_number}\n{text}"
        for page_number, text in page_data
    )

    fingerprint_by_source[source] = hashlib.sha256(
        full_text.encode("utf-8")
    ).hexdigest()


# ============================================
# 6. REMOVE OLD VERSIONS OF UPLOADED DOCUMENTS
# ============================================

uploaded_filenames = set(uploaded.keys())

# Find fingerprints for the documents being uploaded
incoming_fingerprints = {
    fingerprint_by_source[source]
    for source in uploaded_filenames
    if source in fingerprint_by_source
}

# Get existing database entries
existing_data = collection.get(
    include=["metadatas"]
)

ids_to_delete = []

for chunk_id, metadata in zip(
    existing_data["ids"],
    existing_data["metadatas"]
):

    existing_source = metadata.get("source", "")
    existing_fingerprint = metadata.get(
        "document_fingerprint", ""
    )

    # Remove the previous version of this filename
    if existing_source in uploaded_filenames:
        ids_to_delete.append(chunk_id)

    # Also remove copies of the same document under other filenames
    elif existing_fingerprint in incoming_fingerprints:
        ids_to_delete.append(chunk_id)

if ids_to_delete:
    collection.delete(ids=ids_to_delete)

    print(
        "Removed old or duplicate chunks:",
        len(ids_to_delete)
    )


# ============================================
# 7. PREPARE UNIQUE CHUNKS
# ============================================

unique_chunks = []
seen_chunk_hashes = set()

for chunk in chunks:

    source = chunk["source"]

    if source not in fingerprint_by_source:
        continue

    document_fingerprint = fingerprint_by_source[source]

    # Identify a chunk by its document content, page and text
    chunk_hash = hashlib.sha256(
        (
            document_fingerprint
            + "|"
            + str(chunk["page_number"])
            + "|"
            + chunk["text"].strip()
        ).encode("utf-8")
    ).hexdigest()

    # Avoid repeated identical chunks within this upload
    if chunk_hash in seen_chunk_hashes:
        continue

    seen_chunk_hashes.add(chunk_hash)

    unique_chunks.append({
        "text": chunk["text"],
        "source": source,
        "page_number": chunk["page_number"],
        "document_fingerprint": document_fingerprint,
        "chunk_id": chunk_hash
    })


# ============================================
# 8. GENERATE AND STORE EMBEDDINGS
# ============================================

if unique_chunks:

    texts = [
        chunk["text"]
        for chunk in unique_chunks
    ]

    print(
        "Generating embeddings for",
        len(texts),
        "unique chunks..."
    )

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    collection.upsert(
        ids=[
            chunk["chunk_id"]
            for chunk in unique_chunks
        ],
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=[
            {
                "source": chunk["source"],
                "page_number": chunk["page_number"],
                "chunk_id": chunk["chunk_id"],
                "document_fingerprint": (
                    chunk["document_fingerprint"]
                )
            }
            for chunk in unique_chunks
        ]
    )


# ============================================
# 9. COMPLETION STATUS
# ============================================

print("\nBlock 4 completed successfully!")
print("Unique chunks processed:", len(unique_chunks))
print("Total chunks in database:", collection.count())
print("Database location:", DB_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Generating embeddings for 10 unique chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Block 4 completed successfully!
Unique chunks processed: 10
Total chunks in database: 54
Database location: /content/drive/MyDrive/MIT_AOE_RAG_DB


In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 5: HYBRID RETRIEVAL
# ============================================

def retrieve_documents(question, top_k=3):

    # Check whether the database contains any chunks
    total_chunks = collection.count()

    if total_chunks == 0:
        return []

    # ----------------------------------------
    # 1. SEMANTIC SEARCH
    # ----------------------------------------

    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0].tolist()

    # Retrieve chunks with similar meanings
    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, total_chunks),
        include=["documents", "metadatas", "distances"]
    )

    combined_results = {}

    # Store semantic search results
    for i, chunk_id in enumerate(semantic_results["ids"][0]):

        metadata = semantic_results["metadatas"][0][i]
        distance = semantic_results["distances"][0][i]

        combined_results[chunk_id] = {
            "text": semantic_results["documents"][0][i],
            "source": metadata["source"],
            "page_number": metadata["page_number"],
            "chunk_id": chunk_id,
            "semantic_distance": distance,
            "keyword_score": 0
        }

    # ----------------------------------------
    # 2. KEYWORD SEARCH
    # ----------------------------------------

    # Extract words from the question
    question_words = set(
        re.findall(r"\b[a-zA-Z0-9]+\b", question.lower())
    )

    # Remove common words
    stop_words = {
        "what", "is", "are", "the", "a", "an", "of",
        "in", "on", "for", "to", "and", "or", "do",
        "does", "how", "can", "i", "we", "you", "it",
        "about", "tell", "me", "please", "with"
    }

    keywords = question_words - stop_words

    # Get all stored chunks for keyword matching
    all_results = collection.get(
        include=["documents", "metadatas"]
    )

    for i, chunk_id in enumerate(all_results["ids"]):

        text = all_results["documents"][i]
        metadata = all_results["metadatas"][i]

        # Extract words from the stored chunk
        text_words = set(
            re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
        )

        # Calculate how many question keywords match
        matched_words = keywords.intersection(text_words)

        keyword_score = (
            len(matched_words) / len(keywords)
            if keywords else 0
        )

        # Update the score if semantic search found this chunk
        if chunk_id in combined_results:

            combined_results[chunk_id]["keyword_score"] = (
                keyword_score
            )

        # Also include chunks found through keyword matching
        elif keyword_score > 0:

            combined_results[chunk_id] = {
                "text": text,
                "source": metadata["source"],
                "page_number": metadata["page_number"],
                "chunk_id": chunk_id,
                "semantic_distance": None,
                "keyword_score": keyword_score
            }

    # ----------------------------------------
    # 3. COMBINE SCORES AND RANK CHUNKS
    # ----------------------------------------

    ranked_results = []

    for result in combined_results.values():

        # Estimate semantic similarity from cosine distance
        if result["semantic_distance"] is not None:
            semantic_score = max(
                0,
                1 - result["semantic_distance"]
            )
        else:
            semantic_score = 0

        # Combine semantic and keyword scores
        result["combined_score"] = (
            0.7 * semantic_score
            + 0.3 * result["keyword_score"]
        )

        ranked_results.append(result)

    # Sort from highest score to lowest score
    ranked_results.sort(
        key=lambda item: item["combined_score"],
        reverse=True
    )

    # Return the top-ranked chunks
    return ranked_results[:top_k]


# --------------------------------------------
# 4. COMPLETION MESSAGE
# --------------------------------------------

print("Block 5 completed successfully!")
print("Hybrid retrieval function is ready.")
print("Semantic search and keyword ranking are configured.")


Block 5 completed successfully!
Hybrid retrieval function is ready.
Semantic search and keyword ranking are configured.


In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 6: GEMINI CLIENT CONFIGURATION
# ============================================

from google.colab import userdata
from google import genai
import time

# Load the API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Create the Gemini client only once
gemini_client = genai.Client(api_key=api_key)

# Configure the model in one place
GEMINI_MODEL = "gemini-3.8-flash"

print("Block 6 completed successfully!")
print("Gemini client created and ready.")
print("API key loaded securely from Colab Secrets.")
print("Model configuration completed.")


Block 6 completed successfully!
Gemini client created and ready.
API key loaded securely from Colab Secrets.
Model configuration completed.


In [ ]:

# ============================================
# MIT AOE RAG PROJECT
# BLOCK 7: INTERACTIVE RAG CHATBOT
# ============================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import time


# ============================================
# 1. GENERATE ANSWER USING RAG
# ============================================

def ask_rag(question, top_k=3):

    start_time = time.time()

    try:
        # Retrieve relevant information from Block 5
        retrieved_docs = retrieve_documents(
            question,
            top_k=top_k
        )

        if not retrieved_docs:
            return {
                "answer": (
                    "I could not find relevant information "
                    "in the available MIT AOE documents."
                ),
                "sources": [],
                "response_time": round(
                    time.time() - start_time, 2
                )
            }

        # Prepare evidence for Gemini
        context_parts = []

        for index, doc in enumerate(retrieved_docs, start=1):
            context_parts.append(
                f"[Evidence {index}]\n"
                f"Source: {doc['source']}\n"
                f"Page: {doc['page_number']}\n"
                f"Text: {doc['text']}"
            )

        context = "\n\n".join(context_parts)

        # Prepare the prompt
        prompt = f"""
You are an MIT AOE information assistant.

Answer the question using only the evidence below.
Do not invent facts.
If the evidence is insufficient, say so.
Use simple and clear language.

EVIDENCE:
{context}

QUESTION:
{question}

ANSWER:
"""

        # Call Gemini and retry temporary failures
        response = None

        for attempt in range(3):

            try:
                response = gemini_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=prompt
                )
                break

            except Exception as error:

                error_message = str(error).upper()

                temporary_error = any(
                    code in error_message
                    for code in [
                        "503",
                        "UNAVAILABLE",
                        "429",
                        "RESOURCE_EXHAUSTED",
                        "500",
                        "INTERNAL"
                    ]
                )

                # Retry temporary errors only
                if not temporary_error or attempt == 2:
                    raise

                wait_seconds = 2 ** (attempt + 1)

                print(
                    f"Temporary Gemini error. "
                    f"Retrying in {wait_seconds} seconds..."
                )

                time.sleep(wait_seconds)

        # Extract Gemini's response
        answer = (response.text or "").strip()

        if not answer:
            answer = (
                "Gemini returned an empty answer. "
                "Please try again."
            )

        # Collect source references from retrieved metadata
        sources = []
        seen_sources = set()

        for doc in retrieved_docs:

            source_key = (
                doc["source"],
                doc["page_number"]
            )

            if source_key not in seen_sources:
                seen_sources.add(source_key)

                sources.append({
                    "source": doc["source"],
                    "page_number": doc["page_number"]
                })

        return {
            "answer": answer,
            "sources": sources,
            "response_time": round(
                time.time() - start_time, 2
            )
        }

    except Exception as error:

        # Print the actual error for debugging
        print("\nACTUAL ERROR:", repr(error))

        return {
            "answer": (
                "Request failed. Check the actual error "
                "printed above."
            ),
            "sources": [],
            "response_time": round(
                time.time() - start_time, 2
            )
        }


# ============================================
# 2. SIMPLE QUESTION INTERFACE
# ============================================

question_input = widgets.Text(
    placeholder="Ask a question...",
    description="Question:",
    layout=widgets.Layout(width="500px")
)

ask_button = widgets.Button(
    description="Ask"
)

answer_output = widgets.Output()


def handle_question(button):

    question = question_input.value.strip()

    with answer_output:
        clear_output(wait=True)

        if not question:
            print("Please enter a question.")
            return

        print("Generating answer...")

        result = ask_rag(question)

        clear_output(wait=True)

        print(result["answer"])

        if result["sources"]:
            print("\nSources:")

            for source in result["sources"]:
                print(
                    f"- {source['source']}, "
                    f"page {source['page_number']}"
                )

        print(
            f"\nResponse time: "
            f"{result['response_time']} seconds"
        )


ask_button.on_click(handle_question)

display(question_input)
display(ask_button)
display(answer_output)



Text(value='', description='Question:', layout=Layout(width='500px'), placeholder='Ask a question...')

Button(description='Ask', style=ButtonStyle())

Output()